## About
- This notebook includes exploration of model performance vs. number of features in a cross-validation procedure using the discovery cohort

### Input
1. curated proteomics and clinical data

### Output
> MRMR results
> 
> Two-protein panel vs. ALT, AST, and AST/ALT ratio

In [ ]:
# Load packages 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import logging
from tqdm import tqdm
logger = logging.getLogger()
logger.setLevel(logging.INFO)

### Import curated proteomics and clinical data

In [ ]:
# Define project directory
Base = 'data_directory'

df_olink = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='proteomics_curated').set_index('CBMRID')
df_cli = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='clinical_curated').set_index('CBMRID')

In [ ]:
# Join curated proteomics and clinical data
data = df_olink.join(df_cli)
# Drop samples without outcome target measure
data = data[data['cohort']!='GALA_LiHep'].dropna(subset=['inflam_binary'])

In [ ]:
olink_proteins = df_olink.columns
addon = ['alt_log2', 'ast_log2']

In [ ]:
# import MS data 
data_ms = pd.read_csv(os.path.join(Base, 'data_curated/olink_ms_combined.csv')).set_index('CBMRID')
data=data.join(data_ms[['Q08380']])

In [ ]:
olink_ms_proteins = olink_proteins.tolist() + ['Q08380']
olink_ms_proteins_altast = olink_ms_proteins+addon

### Targets

In [ ]:
# rename binary columns
data['I2'] = data['inflam_binary']

In [ ]:
TARGETS = ['I2']
Y = data[TARGETS]
Y['I2'].value_counts(dropna=False)

### Handle missing features of clinical data (Global Missing Pattern)

Features are present to widely different degree. In order to be able to define global splits with the same pattern of missings over the features and targets by samples (here: patients), we define a missing pattern for stratification.

In [ ]:
# Define selected demographic features
SELECTED_DEMOGRAPHICS = ['age', 'gender', 'abstinent', 'bmi']
# Define clinical comparators in the cross-validation procedure
FEATURES_CLINIC = ['ast_log2', 'alt_log2', 'Q08380']

In [ ]:
# Excluding Olink features due to absence of missing values in the fitered data matrix
FEATURES_META_ALL = FEATURES_CLINIC + SELECTED_DEMOGRAPHICS + TARGETS
data[FEATURES_META_ALL].describe().sort_values(by="count", ascending=False, axis=1)

In [ ]:
def ordered_missing_table(data:pd.DataFrame):
    """Order dataframe by data completeness (first column has most features) 
    and then return an encoding of completeness (1 = available, 0 0 not available)"""
       
    data_missing_table = data.notna().astype(int)
    var_ordered_by_completness = list(data.describe().loc['count'].sort_values(ascending=False).index)
    data_missing_table = data_missing_table.sort_values(by=var_ordered_by_completness)[var_ordered_by_completness]
    return data_missing_table.replace(0, pd.NA).convert_dtypes()

print("Used features: {}".format(", ".join(FEATURES_META_ALL)))
data_cli_missing_table = ordered_missing_table(data=data[FEATURES_META_ALL])
data_cli_missing_table = data_cli_missing_table.dropna(how='all', axis=0).dropna(how='all', axis=1)
# data_cli_missing_table #hide

In [ ]:
data_cli_missing_table.describe().loc['count'].astype(int)

In [ ]:
data_cli_missing_table = ordered_missing_table(data_cli_missing_table)

In [ ]:
data_cli_missing_strings = data_cli_missing_table.fillna(value=0)
data_cli_missing_strings = data_cli_missing_strings.astype(str)
stratifier = data_cli_missing_strings.apply(lambda x : x.str.cat(), axis=1)
# display(stratifier.head()) #hide
stratifier_tab = stratifier.value_counts()
stratifier_tab

We will have to get ride of the singletons (unique value only once observed). Possibly the grouping could be extended to the values up to 5.

In [ ]:
unique_missing_patterns = list(stratifier_tab.index)

def match_observed(seq1, seq2):
    return sum(pos1 == pos2 for pos1, pos2 in zip(seq1, seq2))

assert match_observed("11111100", "11110111") == 5, "Failed"

In [ ]:
stratifier.value_counts().min()

In [ ]:
def update_stratifier(stratifier_var:pd.Series, threshold:int=None, verbose:bool=False):
    """Takes a stratifier variable, and assigns the pattern the less 
    often observed (defined by threshold or the minimum) to the closest other missing pattern.
    Clossness is defined by the number of features which are present/absent for the samples."""
    stratifier_var =stratifier_var.copy()
    stratifier_tab = stratifier_var.value_counts()
    current_minimum = stratifier_tab.min()
    if threshold is not None  and current_minimum >= threshold:
        logger.info("Threshold already reached.")
        return stratifier_var
    unique_missing_patterns = list(stratifier_tab.index)
    list_single_missing_patterns = stratifier_tab[stratifier_tab <= current_minimum].index
    for single_missing_pattern in list_single_missing_patterns:
        if verbose:
            logger.info(f"Find match for: {single_missing_pattern}")
        closest = 0
        for i, other_seq in enumerate(unique_missing_patterns):
            if not other_seq in list_single_missing_patterns:
                relatedness = match_observed(single_missing_pattern, other_seq)
                if relatedness > closest:
                    closest = relatedness
                    best = other_seq
        stratifier_var[stratifier_var == single_missing_pattern] = best
        if verbose:
            logger.info(f"Best match is : {best}")
    if threshold is not None:
        stratifier_tab = stratifier_var.value_counts()
        new_minimum = stratifier_tab.min()
        if new_minimum < threshold:
            stratifier_var = update_stratifier(stratifier_var, threshold=threshold)        
    return stratifier_var
# stratifier = update_stratifier(stratifier).value_counts()
stratifier = update_stratifier(stratifier, threshold=5, verbose=True)
stratifier.value_counts()

Global stratification based on string. It won't be possible to distribute unique cases, which is why they were assigned to their closest papern. Then split models between endpoints are comparable as they are subsets of the global splits. This  garuantess:
1. By endpoint: Metrics for marker on one test set does not contain patients which are in the training set of a different marker.
2. By marker: Metrics for different endpoints on the test set does not contain training samples of a different endpoint.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

CV_FOLDS = 5
CV_REPEATS = 10

RANDOM_SEED = 123

rskf = RepeatedStratifiedKFold(n_splits=CV_FOLDS, n_repeats=CV_REPEATS, random_state=RANDOM_SEED) 
splits = list(rskf.split(data_cli_missing_table, stratifier))

In [ ]:
cv_train_test_indices = list()
for train_indices, test_indices in splits:
    cv_train_test_indices.append(
     (stratifier.index[train_indices], stratifier.index[test_indices])
    )
cv_train_test_indices[0]

In [ ]:
from src.cross_validation import run_cv_binary

### mrmr feature selction

In [ ]:
from mrmr import mrmr_classif
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

In [ ]:
clinical_data_to_include={'F2':['ggt', 'ast', 'iga', 'insulin', 'igg'],
                         'F3':['ggt', 'ast', 'iga', 'insulin', 'homair'], 
                         'I2':['ast', 'ggt', 'insulin', 'alt', 'homair'],
                         'S1':['abstinent', 'alt', 'ast', 'homair', 'mets']}
fibrosis_markers=['elf', 'te']

In [ ]:
#data_ml = data[olink_ms_proteins_altast]
data_ml = data[olink_proteins]
discovery_cohorts = ['ALD', 'HP', 'RDC']
validation_cohorts = ['SIP']

train_indices = data[data['cohort'].isin(discovery_cohorts) & data['I2'].notnull()].index
train_indices = list(set(train_indices) & set(data_ml.index))

validation_indices = data[data['cohort'].isin(validation_cohorts)].index
validation_indices = list(set(validation_indices) & set(data_ml.index))

data_ml_train=data_ml.loc[train_indices]
data_ml_validation = data_ml.loc[validation_indices]
data_ml_train_y=data.loc[train_indices]['I2']

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score, roc_curve

In [ ]:
import pickle

In [ ]:
import sklearn

In [ ]:
sklearn.__version__

In [ ]:
RECALCULATE_FEATURES = False
pickle_file_name='final_mrmr_20250628.pkl'
mrmr_path = os.path.join(Base, 'mrmr')
#RESULT_FEATURE_COMPARISON_MRMR = os.path.join('results/mrmr', pickle_file_name)
RESULT_FEATURE_COMPARISON_MRMR = os.path.join(mrmr_path, pickle_file_name)

def n_features_comparison_mrmr(n_features_max=40):
    summary = []
    runs_roc_scores = {}
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
    model = LogisticRegression(random_state=42, solver='liblinear')
    scoring = ['precision', 'recall', 'f1', 'balanced_accuracy', 'roc_auc']
    X = data_ml_train
    y = data_ml_train_y
    in_both = y.index.intersection(X.index)
    _X = X.loc[in_both]
    _y = y.loc[in_both]

    for n_features in tqdm(range(1, n_features_max+1), desc="Evaluating features"):
        results = []
        roc_scores = []
        features = []
        
        for train_index, test_index in cv.split(_X, _y):
            X_train, X_test = _X.iloc[train_index], _X.iloc[test_index]
            y_train, y_test = _y.iloc[train_index], _y.iloc[test_index]

            selected_features = mrmr_classif(X_train, y_train, K=n_features)
            features.append(selected_features)
            
            X_train_sel = X_train[selected_features]
            X_test_sel = X_test[selected_features]

            model.fit(X_train_sel, y_train)
            y_pred = model.predict(X_test_sel)
            y_proba = model.predict_proba(X_test_sel)[:, 1]

            fpr, tpr, thresholds = roc_curve(y_test, y_proba)
            roc_scores.append((fpr, tpr, thresholds))

            fold_result = {
                'test_precision': precision_score(y_test, y_pred, zero_division=0),
                'test_recall': recall_score(y_test, y_pred, zero_division=0),
                'test_f1': f1_score(y_test, y_pred, zero_division=0),
                'test_balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
                'test_roc_auc': roc_auc_score(y_test, y_proba),
                'n_features': n_features,
                'test_case': 'I2',
                'n_observations': _X.shape[0],
            }
            results.append(fold_result)

        summary.append(pd.DataFrame(results))
        runs_roc_scores[n_features]=[roc_scores, features]      

    summary_n_features = pd.concat(summary, ignore_index=True)
    final_result = {'summary_n_features':summary_n_features, 'runs_roc_scores':runs_roc_scores}

    with open(RESULT_FEATURE_COMPARISON_MRMR, 'wb') as f:
        pickle.dump(final_result, f)

    return summary_n_features, runs_roc_scores

# Run or load results
if not RECALCULATE_FEATURES:
    try:
        with open(RESULT_FEATURE_COMPARISON_MRMR, 'rb') as f:
            final_result = pickle.load(f)
        summary_n_features = final_result['summary_n_features']
        runs_roc_scores = final_result['runs_roc_scores']
        
    except FileNotFoundError:
        summary_n_features, runs_roc_scores = n_features_comparison_mrmr()
else:
    summary_n_features, runs_roc_scores = n_features_comparison_mrmr()

#### Aggregate feature selection frequency across all folds. 
> Cross-validation on the discovery cohort only.
> In each fold, apply mRMR to the training set only.
> Select the top n<-(1, 39) features from mRMR in that fold.
> Across all folds, count how often each feature appears in those top n.
> Use this frequency to rank individual features and select a subset (say, top 40) for exhaustive 2-feature combination modeling.
> Aggregating across folds does mean every sample contributes, but because it’s done through training-only views in each fold, you're not violating the principle of no peeking. You're simulating how features would behave if selected on unseen data, which is the whole point of cross-validation.

> This is a standard and accepted practice in machine learning

In [ ]:
from collections import Counter
all_features=[]
for value in runs_roc_scores.values():
    feature_list = value[1]  
    for feature_set in feature_list:
        all_features.extend(feature_set)

In [ ]:
feature_counter = Counter(all_features)
features_all, counts_all = zip(*feature_counter.most_common())
df_feature_freq = pd.DataFrame({'feature':features_all, 'times of selected out of 2000':counts_all})
features, counts = zip(*feature_counter.most_common(n=30))
shortlist = pd.DataFrame({'Protein':features[:30]})

# Plot dot plot (pivoted: features on x-axis, frequency on y-axis)
plt.figure(figsize=(6, 3))
plt.scatter(features, counts, s=50, color='deepskyblue', edgecolor='b')
plt.ylabel('Frequency')
plt.title('Feature Selection Frequency')
plt.xticks(rotation=90)

# Add vertical lines every 10 features
for i in range(0, len(features), 10):
    plt.axvline(x=features[i], color='gray', linestyle='--', linewidth=0.5)
    plt.annotate(i, xy=(features[i], 0))

plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.rcParams['pdf.fonttype'] = 42
plt.tight_layout()
plt.savefig(os.path.join(Base, 'mrmr/ranked_features.png'), dpi=120, bbox_inches='tight')
plt.savefig('temp.pdf', bbox_inches='tight', dpi=120)

#### Write shortlisted proteins based on feature frequency in cross validation

In [ ]:
old_shortlist = pd.read_excel(os.path.join(Base, 'Shortlisting_of_cytokines.xlsx'))
novel=old_shortlist[old_shortlist['novel']==1]['Protein']
# Remove proteins that are known to be no longer novel
novel_nomore = ['CDCP1']
novel = list(set(novel) - set(novel_nomore))
shortlist['novel']=np.where(shortlist['Protein'].isin(novel), True, False)
shortlist.to_csv(os.path.join(Base, 'mrmr/shortlisted_biomarkers_by_mrmr.csv'))

In [ ]:
data_ml_train_y.value_counts()

In [ ]:
combined_mrmr = summary_n_features.groupby(['test_case','n_features']).mean()

In [ ]:
#combined_mrmr.to_csv('results/mrmr/mrmr_combined.csv', )
#summary_n_features.to_csv('results/mrmr/mrmr.csv')

In [ ]:
colors = ['green', 'lightblue', 'blue'] #'red']
labels = ['ROC-AUC', 'F1 score', 'Balanced accuracy', ]#'Recall']
to_plot = ['test_roc_auc', 'test_f1', 'test_balanced_accuracy', ]#'test_recall']

plt.figure(figsize=(4, 4))

for metric, color, label in zip(to_plot, colors, labels):
    sns.lineplot(
        x='n_features',
        y=metric,
        data=summary_n_features[0:2500],
        color=color,
        linewidth=2,
        label=label
    )

plt.ylim([0, 1])
plt.legend(fontsize=10)
#plt.title('Number of Features\nvs. Model Performance', fontsize=14)
plt.xlabel('Number of Features', fontsize=14)
plt.ylabel('Score', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(Base, 'mrmr/mrmr_40features.pdf'), bbox_inches='tight', dpi=120)
plt.savefig('temp.pdf', bbox_inches='tight', dpi=120)
plt.tight_layout()
plt.show()

#### Plot AUROC for 2-feature logistic regression model

In [ ]:
from src.plots import plot_roc_curve

fig, ax = plt.subplots(figsize=(4, 4))      

plot_roc_curve(ax, runs_roc_scores[2][0])
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(Base, 'mrmr/auroc.png'), dpi=120, bbox_inches='tight')
plt.savefig('temp.pdf', bbox_inches='tight', dpi=120)

#### Compare with clinical markers

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
model = LogisticRegression(random_state=42, solver='liblinear')
scoring = ['precision', 'recall', 'f1', 'balanced_accuracy', 'roc_auc']

In [ ]:
df_cli_ml = data[['ast_alt', 'alt', 'ast', 'I2', 'cohort']].dropna()
train_indices = df_cli_ml[df_cli_ml['cohort'].isin(discovery_cohorts)].index

summary_cli=[]
for feature in ['ast_alt', 'ast', 'alt']:
    _X=df_cli_ml.loc[train_indices][[feature]]
    _y=df_cli_ml.loc[train_indices]['I2']
    scores = cross_validate(model, _X, _y, scoring=scoring, cv=cv)
    scores['feature']=feature
    scores['n_features']=1
    scores['test_case']='I2'
    scores['n_observations']=_X.shape[0]
    results=pd.DataFrame(scores)
    summary_cli.append(results)
summary_cli_markers = pd.concat(summary_cli)

In [ ]:
benchmark_olink = summary_n_features[summary_n_features['n_features']==2]
benchmark_olink['feature']='2-marker'
benchmark = pd.concat([benchmark_olink, summary_cli_markers])

In [ ]:
benchmark = benchmark.sort_values(by='test_f1', ascending = False)

In [ ]:
benchmark_long = benchmark[['test_f1', 'test_balanced_accuracy', 'test_roc_auc', 'feature']].melt(id_vars=['feature'])
benchmark_long.groupby(['feature', 'variable'])['value'].mean()

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
sns.barplot(data=benchmark_long, x='variable', y='value', hue='feature', 
            capsize=0.1, errwidth=1, palette='Blues', edgecolor='gray')
ax.legend(bbox_to_anchor=(1.02, 1.0))
plt.ylabel('Score')
plt.xlabel('')
plt.xticks([0, 1, 2], labels=['F1', 'balanced accuracy', 'ROC-AUC'])
plt.rcParams['pdf.fonttype'] = 42
plt.savefig(os.path.join(Base, 'mrmr/benchmark_long.png'), dpi=120, bbox_inches='tight')
plt.savefig('temp.pdf', bbox_inches='tight', dpi=120)

### Save results

In [ ]:
with pd.ExcelWriter('supp_tables/ST3-5.xlsx') as writer:
    summary_n_features.to_excel(writer, sheet_name='ST3', index=False)
    benchmark.drop(['fit_time', 'score_time'], axis=1).to_excel(writer, sheet_name='ST4', index=False)
    df_feature_freq.to_excel(writer, sheet_name='ST5', index=False)